# LangChain Streaming

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/langchain-streaming-202503.ipynb
</div>

## Intro


Streaming means returning an LLM's output incrementally — token by token (or chunk by chunk) — as it's generated, instead of waiting for the entire response to finish before showing anything to the user.

**Why it matters**:
- **Perceived latency**: LLMs can take several seconds (or longer) to generate a full response. Streaming lets users see the first words almost immediately, making the application feel far more responsive even though the total generation time is the same.
- **Better UX for long outputs**: For long-form answers, reports, or code generation, users can start reading and reacting before the response is complete, rather than staring at a blank screen or loading spinner.
- **Early cancellation**: If a response is clearly going off-track, a user (or system) can stop generation early, saving time and cost.
- **Real-time interactivity**: Streaming is essential for chat-like interfaces (e.g., ChatGPT-style UIs), voice assistants, and any application where responsiveness is part of the product experience.

**Where it gets harder:**

Streaming a plain LLM call is straightforward — you just forward tokens as they arrive. But it becomes more complex once you introduce Agents, which run their own internal logic (tool calls, reasoning steps, intermediate "thoughts") before producing a final answer. Naively streaming an agent's raw output exposes all of that internal machinery (JSON actions, tool calls, etc.) instead of just the final, user-facing answer — so extra work is needed to filter and expose only what's relevant.

We'll start simple, streaming LLM output straight to the terminal, then work up to the more advanced case of streaming an Agent's response through a FastAPI backend.


<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [2]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [3]:
# !pip install dotenv "numexpr==2.14.2"

<br>

## Initial Setup



In [4]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


<br>

## LLM Streaming to Stdout

The simplest form of streaming is to print each token as it's generated. To do this, we initialize an LLM (one that supports streaming — not all do) with two parameters:

* `streaming=True` — enables streaming.
* `callbacks=[SomeCallbackHere()]` — a LangChain callback (or list of callbacks) that runs code every time a new token arrives.

Here we use LangChain's built-in `StreamingStdOutCallbackHandler`, which simply prints each token as it's generated.



In [5]:
import os
from langchain_openai import ChatOpenAI
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    model='gpt-6-luna',
    temperature=1,  # gpt-6-luna only accepts the default temperature
    max_completion_tokens=1000,
    streaming=True,  # ! important
    callbacks=[StreamingStdOutCallbackHandler()],  # ! important
)


# LangChain Agents pass a `stop` sequence (e.g. "\nObservation:"). 
# The model that we're using rejects that parameter, 
# so we're adding the custom function below to drop it before the request is sent.
def _payload(self, input_, *, stop=None, **kwargs):
    return _payload.original(self, input_, **kwargs)

_payload.original = getattr(ChatOpenAI._get_request_payload, "original", ChatOpenAI._get_request_payload)
ChatOpenAI._get_request_payload = _payload

/Users/luis/Desktop/ironhack_sept26/demos_ai_engineering_sept2026/.venv/langchain-v0.2/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: UserWarning: WARNING! max_completion_tokens is not default parameter.
                max_completion_tokens was transferred to model_kwargs.
                Please confirm that max_completion_tokens is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


**How it works**

1. `streaming=True` tells OpenAI: **"Don't wait until you've finished the entire answer—send it to me one small piece (token) at a time."**
2. Each time a new token arrives, LangChain immediately passes it to the callback we provided (in this case, `StreamingStdOutCallbackHandler`).

3. `StreamingStdOutCallbackHandler` simply prints each token to the terminal as soon as it receives it.


<br>

Now if we call the LLM, we'll see the response being _streamed_ token by token in the output below, instead of appearing all at once.

In [6]:
llm.invoke("Tell a long story where Willy Wonka invites the world's best AI engineers")

# Willy Wonka and the Thinking Machine

At exactly midnight on the first day of spring, a golden envelope appeared on the desks of the world’s best AI engineers.

It landed beside a coffee mug in Seoul, beneath a stack of research papers in Nairobi, inside a server room in Toronto, and—most mysteriously—on the pillow of an engineer who was already asleep.

Every envelope bore the same words:

> **Willy Wonka requests the pleasure of your company at his chocolate factory.**
>
> **Bring your curiosity. Leave your assumptions.**
>
> **The future is waiting.**
>
> *P.S. Do not bring lunch. Lunch is being handled.*

The engineers came from every corner of the world. There was Dr. Imani Okafor, who built learning systems that helped farmers predict droughts. There was Pavel Sokolov, whose robots could sort a mountain of electronic waste without breaking a single useful part. There was Mina Park, an expert in language models and a champion of asking, “What happens if the answer is wrong?” And

AIMessage(content='# Willy Wonka and the Thinking Machine\n\nAt exactly midnight on the first day of spring, a golden envelope appeared on the desks of the world’s best AI engineers.\n\nIt landed beside a coffee mug in Seoul, beneath a stack of research papers in Nairobi, inside a server room in Toronto, and—most mysteriously—on the pillow of an engineer who was already asleep.\n\nEvery envelope bore the same words:\n\n> **Willy Wonka requests the pleasure of your company at his chocolate factory.**\n>\n> **Bring your curiosity. Leave your assumptions.**\n>\n> **The future is waiting.**\n>\n> *P.S. Do not bring lunch. Lunch is being handled.*\n\nThe engineers came from every corner of the world. There was Dr. Imani Okafor, who built learning systems that helped farmers predict droughts. There was Pavel Sokolov, whose robots could sort a mountain of electronic waste without breaking a single useful part. There was Mina Park, an expert in language models and a champion of asking, “What h

<br>

That was surprisingly easy — but streaming gets more complicated once we introduce agents, which reason over multiple steps and call tools before producing a final answer. Let's initialize an agent to see why.

In [7]:
from langchain.memory import ConversationBufferWindowMemory
from langchain.agents import load_tools, AgentType, initialize_agent
from langchain.tools import Tool

# initialize conversational memory
memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    k=5,
    return_messages=True,
    output_key="output"
)

# create tools for the agent
# (we'll use LangChain's built-in LLM math tool and a custom shipping lookup)
def lookup_shipping_days(country: str) -> str:
    """Returns an estimated shipping time for a destination country."""
    shipping = {
        "Spain": "2–3 business days",
        "France": "3–4 business days",
        "Japan": "5–8 business days",
    }
    return shipping.get(country.strip().title(), "Shipping estimate unavailable.")

tools = load_tools(["llm-math"], llm=llm)

tools.append(
    Tool(
        name="lookup_shipping_days",
        func=lookup_shipping_days,
        description="Look up the estimated shipping time in business days for a destination country. Input a country name, such as Spain, France, or Japan.",
    )
)

# initialize the agent
agent = initialize_agent(
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    tools=tools,
    llm=llm,
    memory=memory,
    verbose=False, # set to True for detailed logging of agent steps
    max_iterations=3,
    early_stopping_method="generate",
    return_intermediate_steps=False
)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_7729/3344696408.py:35: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


<br>

The agent reuses the same `llm` object, so it already has our `StreamingStdOutCallbackHandler` attached. Let's see what streaming looks like when running the agent.

In [8]:
prompt = "Hello, how are you?"

result_1 = agent(prompt)

# print(result_1) # we can keep this commented ('StreamingStdOutCallbackHandler' will automatically print each token in the terminal)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_7729/150748627.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result_1 = agent(prompt)


```json
{
  "action": "Final Answer",
  "action_input": "Hello! I’m doing well, thanks for asking. How are you?"
}
```

<br>

Not bad, but now we're streaming the LLM's _entire_ raw output — including the JSON structure the agent uses internally to decide on tool calls and reasoning steps. This is useful for debugging, but not something we'd want to show end users. 

Let's see another example where we ask a math question that requires a tool call:

In [9]:
result_2 = agent("I want to purchase a product, how long will it take to ship to Spain?")

# print(result_2)

```json
{
  "action": "lookup_shipping_days",
  "action_input": "Spain"
}
``````json
{
  "action": "Final Answer",
  "action_input": "Shipping to Spain is estimated to take 2–3 business days."
}
```

Looking at the streamed output above, we can see the raw JSON "thoughts" the agent streams token-by-token as it works through the problem.

This is exactly the problem we mentioned earlier: streaming shows us all of this internal JSON reasoning, when really we'd only want to stream the final answer text to an end user.

<br>

The raw output can be very useful in some cases (e.g. debugging), but in many other cases you only want to stream the final answer. 

We have two options: 
- Write a custom callback handler
- Use LangChain's purpose-built `FinalStreamingStdOutCallbackHandler`. 

<br>

Let's try the built-in one first. Before changing anything, let's just inspect the LLM's current `callbacks` (still our `StreamingStdOutCallbackHandler` from before):

In [10]:
from langchain.callbacks.streaming_stdout_final_only import FinalStreamingStdOutCallbackHandler

# Configure the LLM to begin streaming only after the model emits the
# tokens ["Final", "Answer"].
agent.agent.llm_chain.llm.callbacks = [
    FinalStreamingStdOutCallbackHandler(
        answer_prefix_tokens=["Final", "Answer"]
    )
]

Let's try it out:

In [11]:
result_3 = agent("I want to purchase a product, how long will it take to ship to Spain?")

# print(result_3)

",
  "action_input": "Shipping to Spain is estimated to take 2–3 business days."
}
```

<br>

Still not quite clean — tuning `answer_prefix_tokens` correctly is finicky. It's usually easier to write a custom callback that filters tokens ourselves, only printing once it detects we're inside the `action_input` of the final answer:

In [12]:
import re
import sys
from langchain.callbacks.base import BaseCallbackHandler

class FinalAnswerStreamingCallbackHandler(BaseCallbackHandler):
    def __init__(self):
        self._reset()

    def _reset(self):
        self.content = ""
        self.final_answer = False
        self.in_action_input = False
        self.escaped = False
        self.unicode_escape = None
        self.answer_done = False

    def on_llm_start(self, serialized, prompts, **kwargs):
        self._reset()

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self._reset()

    def on_llm_new_token(self, token, **kwargs):
        if self.answer_done:
            return

        if not self.final_answer:
            self.content += token
            marker = "Final Answer"
            marker_index = self.content.find(marker)
            if marker_index == -1:
                self.content = self.content[-(len(marker) - 1):]
                return
            self.content = self.content[marker_index + len(marker):]
            self.final_answer = True
        else:
            self.content += token

        if not self.in_action_input:
            match = re.search(r'"action_input"\s*:\s*"', self.content)
            if match is None:
                return
            self.content = self.content[match.end():]
            self.in_action_input = True

        for character in self.content:
            if self.unicode_escape is not None:
                self.unicode_escape += character
                if len(self.unicode_escape) == 4:
                    sys.stdout.write(chr(int(self.unicode_escape, 16)))
                    self.unicode_escape = None
                    self.escaped = False
            elif self.escaped:
                if character == "u":
                    self.unicode_escape = ""
                    self.escaped = False
                else:
                    escaped_characters = {
                        '"': '"',
                        "\\": "\\",
                        "/": "/",
                        "b": "\b",
                        "f": "\f",
                        "n": "\n",
                        "r": "\r",
                        "t": "\t",
                    }
                    sys.stdout.write(escaped_characters.get(character, character))
                    self.escaped = False
            elif character == "\\":
                self.escaped = True
            elif character == '"':
                self.answer_done = True
                break
            else:
                sys.stdout.write(character)

        sys.stdout.flush()
        self.content = ""

callback_handler = FinalAnswerStreamingCallbackHandler()
agent.agent.llm_chain.llm.callbacks = [callback_handler]

<br>

Let's try again with our custom handler:

In [14]:
result_4 = agent("I want to purchase a product, how long will it take to ship to Spain? Also, after answering that question, add please a line break, and 5 paragraphs with a story of Willy Wonka")

# print(result_4)

Shipping to Spain is estimated to take 2–3 business days.

One morning, Willy Wonka discovered a tiny golden door hidden behind a mountain of cocoa beans. A note on it read: “For the curious, and only the curious.”

He turned the little knob, and the door opened onto a garden filled with peppermint trees and chocolate streams. In the middle stood a shy, mechanical fox whose winding key had gone missing.

Willy searched among the candy flowers until he spotted the key glimmering beneath a gumdrop. He returned it to the fox, who sprang to life and led him to a secret grove of singing lollipops.

The lollipops sang a tune so delightful that Willy decided to make a new candy inspired by their melody. He called it the Whistling Wonder, and every piece played a different cheerful note.

That evening, Willy shared the first box with the factory workers. Soon the whole factory was humming together, while the little golden door waited quietly for its next curious visitor.